<a href="https://colab.research.google.com/github/tejuuu-7774/GENAI_collabs/blob/main/6_04_Essay_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#The Strategic Researcher

In [1]:
# # 1. Install dependencies
# !pip install -qU langchain-groq langgraph

# import os
# from typing import TypedDict, List
# from google.colab import userdata
# from langchain_groq import ChatGroq
# from langchain_core.prompts import ChatPromptTemplate
# from langgraph.graph import StateGraph, START, END

# # 2. Setup API Key from Secrets
# # Using your specific secret name: genai_lab_key
# os.environ["GROQ_API_KEY"] = userdata.get('genai_lab_key')

# # Initialize LLM
# llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.1)

# # 3. Define the Global State
# class AgentState(TypedDict):
#     topic: str
#     key_terms: List[str]
#     search_results: List[str]

# # 4. Node: Strategic Researcher
# def research_node(state: AgentState):
#     topic = state["topic"]
#     print(f"--- STEP 1: GENERATING STRATEGIC KEY TERMS FOR: {topic} ---")

#     prompt = ChatPromptTemplate.from_messages([
#         ("system", """You are an Academic Research Lead.
#         Break down the topic into 5 distinct, high-impact key terms.
#         Cover: Technical definitions, historical context, and major ethical debates.
#         Return ONLY a comma-separated list. No intro, no bullets."""),
#         ("human", "{topic}")
#     ])

#     response = (prompt | llm).invoke({"topic": topic})
#     terms = [t.strip().replace('"', '') for t in response.content.split(",")]

#     return {"key_terms": terms[:5]}

# # 5. Build Step 1 Graph
# step1_builder = StateGraph(AgentState)
# step1_builder.add_node("researcher", research_node)
# step1_builder.add_edge(START, "researcher")
# step1_builder.add_edge("researcher", END)
# step1_app = step1_builder.compile()

# # Test Step 1
# initial_input = {"topic": "The impact of renewable energy on global geopolitics"}
# step1_output = step1_app.invoke(initial_input)

# print("\nStep 1 Result (Key Terms):", step1_output['key_terms'])

#The Web Search Tool

In [2]:
# # 1. Install both the integration and the community tools
# !pip install -qU langchain-tavily langchain-community

# import os
# from google.colab import userdata
# # Updated import path to ensure compatibility
# from langchain_community.tools.tavily_search import TavilySearchResults

# # 2. Setup Search Tool using your secrets
# os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')
# # If the above import still fails, use this alternative:
# # from langchain_tavily import TavilySearch as TavilySearchResults

# search_tool = TavilySearchResults(max_results=2)

# # 3. Node: Web Searcher (Same logic, stable import)
# def search_node(state: AgentState):
#     print(f"--- STEP 2: SEARCHING WEB FOR {len(state['key_terms'])} TERMS ---")

#     all_snippets = []
#     for term in state['key_terms']:
#         print(f"Searching for: {term}...")
#         try:
#             results = search_tool.invoke({"query": term})
#             # Extract content from search results
#             for res in results:
#                 all_snippets.append(res['content'])
#         except Exception as e:
#             print(f"Error searching for {term}: {e}")

#     return {"search_results": all_snippets}

# # 4. Re-compile the full workflow with the fix
# full_workflow = StateGraph(AgentState)
# full_workflow.add_node("researcher", research_node)
# full_workflow.add_node("searcher", search_node)

# full_workflow.add_edge(START, "researcher")
# full_workflow.add_edge("researcher", "searcher")
# full_workflow.add_edge("searcher", END)

# app = full_workflow.compile()

# # 5. Run it again
# final_state = app.invoke({"topic": "The impact of renewable energy on global geopolitics"})
# print(f"\nSUCCESS: Gathered {len(final_state['search_results'])} research snippets.")

#The Synthesis Draft Writer

In [3]:
# # 1. Update State (Ensure 'draft' is included)
# class AgentState(TypedDict):
#     topic: str
#     key_terms: List[str]
#     search_results: List[str]
#     draft: str  # This is the target for Step 3

# # 2. Node: The First Draft Writer
# def writer_node(state: AgentState):
#     print(f"--- STEP 3: SYNTHESIZING RESEARCH INTO FIRST DRAFT ---")

#     # We join all the snippets we found in Step 2 into one big context block
#     research_context = "\n\n".join(state["search_results"])

#     prompt = ChatPromptTemplate.from_messages([
#         ("system", """You are a Senior Technical Writer.
#         Your goal is to transform raw research snippets into a high-quality first draft.

#         CRITICAL INSTRUCTIONS:
#         1. STRUCTURE: Create a compelling title, an introduction with a hook, 3 clear body paragraphs, and a forward-looking conclusion.
#         2. FLOW: Use transitional phrases (e.g., "Furthermore," "In contrast to," "Building on this concept") to ensure smooth movement between ideas.
#         3. ACCURACY: Use the provided Research Context to support every claim.
#         4. TONE: Maintain a professional, academic, yet accessible tone.

#         Do not include any intro/outro chatter. Just the essay."""),
#         ("human", f"TOPIC: {state['topic']}\n\nRESEARCH CONTEXT:\n{research_context}")
#     ])

#     # We use a lower temperature (0.2) to keep the writing factual and grounded
#     chain = prompt | llm
#     response = chain.invoke({})

#     return {"draft": response.content}

# # 3. Update and Test the Graph (Connecting 1, 2, and 3)
# builder = StateGraph(AgentState)

# builder.add_node("researcher", research_node)
# builder.add_node("searcher", search_node)
# builder.add_node("writer", writer_node)

# builder.add_edge(START, "researcher")
# builder.add_edge("researcher", "searcher")
# builder.add_edge("searcher", "writer")
# builder.add_edge("writer", END)

# app = builder.compile()

# # 4. Run it
# final_output = app.invoke({"topic": "The impact of NVIDIA's Blackwell architecture on AI development"})

# print("\n" + "="*30)
# print("DRAFT PREVIEW:")
# print(final_output['draft'])

#The Editorial Quality Review

In [4]:
# # 1. Update State to include 'critique'
# class AgentState(TypedDict):
#     topic: str
#     key_terms: List[str]
#     search_results: List[str]
#     draft: str
#     critique: str  # This is the target for Step 4

# # 2. Node: The Editorial Critic
# def reviewer_node(state: AgentState):
#     print(f"--- STEP 4: REVIEWING DRAFT FOR EXCELLENT QUALITY ---")

#     # We provide the critic with the draft AND the original research to check for accuracy
#     research_context = "\n\n".join(state["search_results"])

#     prompt = ChatPromptTemplate.from_messages([
#         ("system", """You are a Senior Editor at a prestigious academic journal.
#         Your goal is to provide a rigorous, high-level critique of the essay draft.

#         EVALUATION CRITERIA:
#         1. ARGUMENT: Is the thesis clear? Does it follow a logical progression?
#         2. EVIDENCE: Does it effectively use the provided research context? Are there any factual gaps?
#         3. FLOW: Are the transitions between paragraphs smooth or jarring?
#         4. TONE: Is the language sophisticated and appropriate for the topic?

#         OUTPUT FORMAT:
#         Provide a list of 3-5 specific, actionable "Redline" improvements.
#         Be direct and critical. Do not praise the draft; only identify what needs to change for 'Excellent Quality'."""),
#         ("human", f"DRAFT TO REVIEW:\n{state['draft']}\n\nORIGINAL RESEARCH CONTEXT:\n{research_context}")
#     ])

#     # We use a slightly higher temperature (0.3) to allow for more creative "critical thinking"
#     chain = prompt | llm
#     response = chain.invoke({})

#     return {"critique": response.content}

# # 3. Update the Graph (Connecting 1 -> 2 -> 3 -> 4)
# builder = StateGraph(AgentState)

# builder.add_node("researcher", research_node)
# builder.add_node("searcher", search_node)
# builder.add_node("writer", writer_node)
# builder.add_node("reviewer", reviewer_node)

# builder.add_edge(START, "researcher")
# builder.add_edge("researcher", "searcher")
# builder.add_edge("searcher", "writer")
# builder.add_edge("writer", "reviewer") # New connection
# builder.add_edge("reviewer", END)

# app = builder.compile()

# # 4. Run the 4-Step Chain
# result = app.invoke({"topic": "The future of autonomous vehicles in urban logistics"})

# print("\n" + "="*50)
# print("EDITORIAL CRITIQUE (STEP 4):")
# print(result['critique'])

#The Executive Finalizer

In [5]:
# # 1. Update State to include 'final_essay'
# class AgentState(TypedDict):
#     topic: str
#     key_terms: List[str]
#     search_results: List[str]
#     draft: str
#     critique: str
#     final_essay: str  # The final output for Step 5

# # 2. Node: The Master Finalizer
# def finalizer_node(state: AgentState):
#     print(f"--- STEP 5: GENERATING FINAL POLISHED ESSAY ---")

#     prompt = ChatPromptTemplate.from_messages([
#         ("system", """You are a Master Editor and Lead Author.
#         Your task is to take a first draft and a list of professional critiques to produce a perfect final essay.

#         EXECUTION STEPS:
#         1. Read the Critique carefully.
#         2. Rewrite the Draft to fix every single issue mentioned in the critique.
#         3. Ensure the 'Flow' is seamless and the vocabulary is elevated.
#         4. Remove any meta-commentary (do not say "Here is the revised version").
#         5. Output ONLY the final essay text, formatted with a clear title and paragraphs."""),
#         ("human", f"ORIGINAL DRAFT:\n{state['draft']}\n\nEDITORIAL CRITIQUE:\n{state['critique']}")
#     ])

#     # We keep temperature very low (0.1) for maximum precision in following edits
#     chain = prompt | llm
#     response = chain.invoke({})

#     return {"final_essay": response.content}

# # 3. Build the COMPLETE 5-Step Graph
# final_builder = StateGraph(AgentState)

# # Add all nodes from our journey
# final_builder.add_node("researcher", research_node) # Step 1
# final_builder.add_node("searcher", search_node)     # Step 2
# final_builder.add_node("writer", writer_node)       # Step 3
# final_builder.add_node("reviewer", reviewer_node)   # Step 4
# final_builder.add_node("finalizer", finalizer_node) # Step 5

# # Define the full sequence
# final_builder.add_edge(START, "researcher")
# final_builder.add_edge("researcher", "searcher")
# final_builder.add_edge("searcher", "writer")
# final_builder.add_edge("writer", "reviewer")
# final_builder.add_edge("reviewer", "finalizer")
# final_builder.add_edge("finalizer", END)

# # Compile the full application
# essay_bot = final_builder.compile()

# # 4. Run the Full Agent
# final_result = essay_bot.invoke({"topic": "The socio-economic impact of Universal Basic Income in developing nations"})

# print("\n" + "="*60)
# print("FINAL ESSAY (STEP 5):")
# print("="*60 + "\n")
# print(final_result['final_essay'])